In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys

import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')

print('Importing the values of the constants...')
# Add the directory containing Constants to the system path
if ('./Constants' not in sys.path):
    sys.path.append('./Constants')  # project root
# Import the constants
import CTE

print('Importing function to load the DataFrames...')
# Add the directory containing LoadDataFrames to the system path
if ('./DataFrameUtils' not in sys.path):
    sys.path.append('./DataFrameUtils')  # project root
from Files import filename_to_dataframe
from DFCleaning import *

print('Importing Functions to make cut masks...')
# Add the directory containing Constants to the system path
if ('./CutMasks' not in sys.path):
    sys.path.append('./CutMasks')  # project root
# Import the constants
import CutMasks

print('Importing Helper Funcions...')
# Add the directory containing Constants to the system path
if ('./HelperFunctions' not in sys.path):
    sys.path.append('./HelperFunctions')  # project root
# Import the constants
import HelperFunctions

print('Importing make_df...')
# Add the directory containing Constants to the system path
if ('./makedf' not in sys.path):
    sys.path.append('./makedf')  # project root
# Import the constants
import make_cc1pidf

# Load DataFrames

In [ ]:
## Check keys in each file
test_file = "../../test_data/test.df"
print("keys in test_file")
splh.print_keys(test_file)

## Check split multiplicity
print("mc_bnb_cosmic_file n_split: %d" %splh.get_n_split(test_file))

In [ ]:
## Define keys to load
print('MC dataframes')
n_max_concat = 10 ## for big files, each key could have more than one split
keys2load = ["evt", "hdr", "histpotdf", "mcnu", "hit0", "hit1", "hit2"] ## keys from the configuration file
test_df = splh.load_dfs(test_file, keys2load, n_max_concat)
print('test data loaded!')

In [ ]:
test_df['mcnu'].columns

# Filter DataFrame

In [ ]:
#Perform duplication validation
print("duplication for Spring Production BNB + Cosmic sample")
find_duplicate_run_evt_combinations(test_df['hdr'])

In [ ]:
plot_duplicate_run_subrun_evt_distribution(test_df["hdr"], "test_df")

In [ ]:
### Filter the hdr DataFrame first, then filter other DataFrames by matching with the hdr DataFrame
test_df["hdr"] = filter_unique_events(test_df["hdr"])
find_duplicate_run_evt_combinations(test_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    test_df[key] = filter_using_hdr(test_df[key], test_df["hdr"])

# Check normalization

In [ ]:
def get_n_evt(df):
    unique_count = df.index.droplevel(
        list(df.index.names[2:])  # drop everything except first two levels
    ).nunique()
    return unique_count

In [ ]:
## Collect the offbeam data fudge factor and scale for offbeam data
#n_record_spill_data = get_n_evt(data_bnb_light_dfs['hdr'])
#n_gates_data = len(data_bnb_light_dfs["pot"])

#n_record_spill_offbeam_data = get_n_evt(data_offbeam_light_dfs['hdr'])
#n_gates_offbeam_data = data_offbeam_light_dfs["hdr"][data_offbeam_light_dfs["hdr"]['first_in_subrun'] == 1]['noffbeambnb'].sum()

#p_trig_data = n_record_spill_data / n_gates_data
#p_trig_offbeam_data = n_record_spill_offbeam_data / n_gates_offbeam_data

#f_factor = (p_trig_data - p_trig_offbeam_data) / (1 - p_trig_offbeam_data)
#print("f_factor: %f" %f_factor)

#intime_gate_scale = (1. - f_factor) * (n_gates_data + 0.) / (n_gates_offbeam_data + 0.)
#print("intime_gate_scale: %f" %intime_gate_scale)

In [ ]:
## Collect pot scale for MC
mc_tot_pot = test_df["hdr"]['pot'].sum()
#mc_low_th_tot_pot = mc_rockbox_th1to100_dfs["hdr"]['pot'].sum()

data_tot_pot = 3.63462e+18
#data_tot_pot = data_bnb_light_dfs["hdr"]['pot'].sum()
#data_tot_TOR860 = data_bnb_light_dfs["pot"]['TOR860'].sum()
#data_tot_TOR875 = data_bnb_light_dfs["pot"]['TOR875'].sum()

print("mc_tot_pot: %e" %(mc_tot_pot))
#print("mc_low_thtot_pot: %e" %(mc_low_th_tot_pot))

#print("data_tot_pot: %e" %(data_tot_pot))
#print("data_tot_TOR860: %e" %(data_tot_TOR860))
#print("data_tot_TOR875: %e" %(data_tot_TOR875))

target_pot = data_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
#mc_low_th_scale = target_pot / mc_low_th_tot_pot
print("MC POT scale: %.3f" %(mc_pot_scale))
#print("MC Low Th. POT scale: %.3f" %(mc_low_th_scale))

In [ ]:
## Comparison between observed and expected total number of recorded spills
n_evt_mc = get_n_evt(test_df["hdr"])
#n_evt_mc_low_th = get_n_evt(mc_rockbox_th1to100_dfs["hdr"])

#print("n_evt_data_onbeam: %d" %n_record_spill_data)
#print("n_evt_exp.: %f" %(n_evt_mc * mc_pot_scale + n_evt_mc_low_th * mc_low_th_scale +n_record_spill_offbeam_data * intime_gate_scale))
print("- n_evt_mc: %f" %(n_evt_mc * mc_pot_scale))
#print("- n_evt_mc_low_th: %f" %(n_evt_mc_low_th * mc_low_th_scale))
#print("- n_evt_data_offbeam: %f" %(n_record_spill_offbeam_data * intime_gate_scale))

# Test background composition

In [ ]:
slc_df = test_df["evt"]
print(len(slc_df.slc.truth.columns))
slc_df = make_cc1pidf.add_nu_categ_column(slc_df)
print(len(slc_df.slc.truth.columns))

In [ ]:
truth_df = slc_df.slc.truth
print(truth_df[truth_df.nu_categ == "none"].npi_P_85MeV_10000MeV)
print(truth_df[truth_df.nu_categ == "none"].np_P_325MeV_10000MeV)
print(truth_df[truth_df.nu_categ == "none"].npi_P_130MeV_800MeV)

In [ ]:
HelperFunctions.print_purity(slc_df)

In [ ]:
obvious_cosmic_mask = CutMasks.is_obvious_cosmic_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask])

In [ ]:
t0_mask = CutMasks.t0_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask])

In [ ]:
is_inside_FV_mask = CutMasks.is_inside_FV_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask])

In [ ]:
nu_score_mask = CutMasks.nu_score_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask])

In [ ]:
track_mask = CutMasks.track_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask])

In [ ]:
shower_mask = CutMasks.shower_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask])

In [ ]:
slc_df = make_cc1pidf.add_best_chi2_columns(slc_df)
chi2_mask = CutMasks.chi2_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask])

In [ ]:
slc_df = make_cc1pidf.add_angle_between_candidates_column(slc_df)
angle_mask = CutMasks.angle_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask & angle_mask])

In [ ]:
containment_mask = CutMasks.containment_cut_mask(slc_df)
HelperFunctions.print_purity(slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & track_mask & shower_mask & chi2_mask & angle_mask & containment_mask])

In [ ]:
pd.set_option('display.max_columns', None)
evdf = test_df["evt"]

#evdf.slc.truth.pdg
for col in evdf.slc.truth.columns:
    print(col)